# Doubled Haploid Tutorial - Python Version

This notebook replicates the AlphaSimR **Doubled Haploid (DH)** line-breeding tutorial using **AlphaSimPy**.
It demonstrates a phenotypic line-breeding program using doubled haploid technology to create homozygous lines instantly, followed by multi-stage yield trials.

**Scenario**: `LinePheno_DH` (phenotypic selection with DH technology)

**Based on R tutorial files**:
- `GlobalParameters.R`
- `CreateParents.R`
- `FillPipeline.R`
- `AdvanceYear.R`
- `UpdateParents.R`
- `ANALYZERESULTS.R`

**Authors**: Translated from AlphaSimR tutorial by Jon Bancic, Philip Greenspoon, Chris Gaynor, Gregor Gorjanc  
**Python translation**: AlphaSimPy project  
**Package**: AlphaSimPy


## Import Required Libraries


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    makeDH,
    setPheno,
    selectInd,
    selectWithinFam,
    meanG,
    varG,
    mergePops,
)

print("AlphaSimPy Doubled Haploid (DH) Tutorial")
print("All libraries imported successfully!")


## Global Parameters

Set up the simulation parameters for the DH line-breeding program.

This closely follows `GlobalParameters.R` in the R tutorial.


In [ ]:
# Number of simulation replications and breeding cycles
n_reps = 1          # Number of simulation replicates
n_burnin = 20       # Number of years in burnin phase
n_future = 20       # Number of years in future phase
n_cycles = n_burnin + n_future
start_tp = 18       # Year to start training population (for future GS tutorials)

# Genome simulation
n_chr = 10          # Number of chromosomes
n_qtl = 1000        # Number of QTL per chromosome
n_snp = 400         # Number of SNP per chromosome

# Initial parents mean and variance
init_mean_g = 1.0   # Initial mean genetic value
init_var_g = 1.0    # Initial genetic variance
init_var_env = 1e-6 # Virtually zero for consistency with 2-Part paper
init_var_gxe = 2.0  # GxE variance
var_e = 4.0         # Yield trial error variance (bushels per acre)
                    # Relates to error variance for an entry mean

# Breeding program details
n_parents = 50      # Number of parents to start a breeding cycle
n_crosses = 100     # Number of crosses per year
n_dh = 100          # DH lines produced per cross
fam_max = 10        # Maximum number of DH lines per cross to enter PYT
n_PYT = 500         # Entries per preliminary yield trial
n_AYT = 50          # Entries per advanced yield trial
n_EYT = 10          # Entries per elite yield trial

# Effective replication of yield trials (controls h2 at each stage)
rep_HDRW = 4.0 / 9.0  # h2 ≈ 0.1
rep_PYT = 1.0         # h2 ≈ 0.2
rep_AYT = 4.0         # h2 ≈ 0.5
rep_EYT = 8.0         # h2 ≈ 0.7

print("Simulation Parameters")
print(f"  Replicates: {n_reps}")
print(f"  Burn-in years: {n_burnin}")
print(f"  Future years: {n_future}")
print(f"  Total cycles: {n_cycles}")
print(f"  Chromosomes: {n_chr}")
print(f"  QTL per chromosome: {n_qtl}")
print(f"  SNP per chromosome: {n_snp}")
print(f"  Parents per cycle: {n_parents}")
print(f"  Crosses per year: {n_crosses}")
print(f"  DH lines per cross: {n_dh}")
print(f"  famMax (max lines per family into PYT): {fam_max}")
print(f"  PYT / AYT / EYT sizes: {n_PYT} / {n_AYT} / {n_EYT}")


## Create Founders and Initial Parents

This mirrors `CreateParents.R`: simulate founder haplotypes, set up `SimParam`, define a trait with GxE, enable pedigree tracking, create founder parents, and assign initial EYT-like phenotypes.


In [ ]:
print("Creating founders and initial parents...")

# Generate initial haplotypes (founders)
founder_pop = runMacs(
    nInd=n_parents,
    nChr=n_chr,
    segSites=n_qtl + n_snp,
    inbred=True,
    species="WHEAT",
)

# Simulation parameters
SP = SimParam(founder_pop)

# Restrict segregating sites (QTL vs SNP)
SP.restrSegSites(nQtlPerChr=n_qtl, minSnpPerChr=n_snp)

# Add SNP chip
if n_snp > 0:
    SP.addSnpChip(n_snp)

# Add trait with GxE and small environmental variance
SP.addTraitAG(
    nQtlPerChr=n_qtl,
    mean=init_mean_g,
    var=init_var_g,
    varEnv=init_var_env,
    varGxE=init_var_gxe,
)

# Enable pedigree tracking
SP.setTrackPed(True)

# Create founder parents
Parents = newPop(founder_pop, sim_param=SP)

# Add phenotype reflecting evaluation in EYT
Parents = setPheno(Parents, varE=var_e, reps=rep_EYT, simParam=SP)

print(f"✓ Founders created: {founder_pop.n_ind} individuals, {founder_pop.n_loci[0]} loci")
print(f"✓ Parents created: {Parents.n_ind} individuals, {Parents.n_traits} traits")
print(f"  Mean G: {meanG(Parents)[0]:.3f}, Var G: {varG(Parents)[0]:.3f}")


## Fill Breeding Pipeline

This cell mirrors `FillPipeline.R`. It fills the breeding pipeline with unique individuals from initial parents, working through stages:

1. **Stage 1**: F1 crosses from parents
2. **Stage 2**: DH lines from F1 (using `makeDH()` - key difference from SSD)
3. **Stage 3**: HDRW (headrow) evaluation
4. **Stage 4**: PYT (preliminary yield trial)
5. **Stage 5**: AYT (advanced yield trial)
6. **Stage 6**: EYT (elite yield trial)

The pipeline is filled by running cohorts 1-7, where each cohort advances one stage.


In [ ]:
print("Filling breeding pipeline with DH technology...")

# Initialize variables
F1 = None
DH = None
HDRW = None
PYT = None
AYT = None
EYT = None

# Fill pipeline with unique individuals from initial parents
for cohort in range(1, 8):
    print(f"  FillPipeline stage: {cohort} of 7")
    
    # Stage 1: F1 crosses
    if cohort < 7:
        F1 = randCross(Parents, n_crosses, simParam=SP)
    
    # Stage 2: DH lines from F1 (using makeDH instead of self)
    if cohort < 6:
        DH = makeDH(F1, nDH=n_dh, simParam=SP)
    
    # Stage 3: HDRW evaluation
    if cohort < 5:
        HDRW = setPheno(DH, varE=var_e, reps=rep_HDRW, simParam=SP)
    
    # Stage 4: PYT
    if cohort < 4:
        PYT = selectWithinFam(HDRW, nInd=fam_max, simParam=SP)
        PYT = selectInd(PYT, nInd=n_PYT, simParam=SP)
        PYT = setPheno(PYT, varE=var_e, reps=rep_PYT, simParam=SP)
    
    # Stage 5: AYT
    if cohort < 3:
        AYT = selectInd(PYT, nInd=n_AYT, simParam=SP)
        AYT = setPheno(AYT, varE=var_e, reps=rep_AYT, simParam=SP)
    
    # Stage 6: EYT
    if cohort < 2:
        EYT = selectInd(AYT, nInd=n_EYT, simParam=SP)
        EYT = setPheno(EYT, varE=var_e, reps=rep_EYT, simParam=SP)

print("Pipeline filled.")
if DH is not None:
    print(f"  DH lines: {DH.n_ind} individuals")
if EYT is not None:
    print(f"  EYT entries: {EYT.n_ind} individuals")


## Advance Year Function

This function mirrors `AdvanceYear.R`. It advances the breeding pipeline by one year, working backwards through stages (EYT → AYT → PYT → HDRW → DH → F1). This is called each year during burn-in and future phases.

**Key feature**: Uses `makeDH()` to create doubled haploid lines from F1 crosses, instantly achieving homozygosity without multiple generations of selfing.


In [ ]:
def advance_year(Parents, F1, DH, HDRW, PYT, AYT, EYT, year, output):
    """
    Advance breeding program by 1 year.
    
    Works backwards through pipeline to avoid copying data.
    Mirrors AdvanceYear.R logic.
    """
    # Stage 6: EYT
    EYT = selectInd(AYT, nInd=n_EYT, simParam=SP)
    EYT = setPheno(EYT, varE=var_e, reps=rep_EYT, simParam=SP)
    
    # Stage 5: AYT
    AYT = selectInd(PYT, nInd=n_AYT, simParam=SP)
    AYT = setPheno(AYT, varE=var_e, reps=rep_AYT, simParam=SP)
    
    # Stage 4: PYT
    # Calculate selection accuracy (correlation between G and P)
    if HDRW is not None and HDRW.n_ind > 0:
        gv_array = np.array(HDRW.gv)[:, 0]
        pheno_array = np.array(HDRW.pheno)[:, 0]
        if len(gv_array) > 1 and np.std(pheno_array) > 0:
            output['accSel'][year-1] = np.corrcoef(gv_array, pheno_array)[0, 1]
        else:
            output['accSel'][year-1] = np.nan
    
    PYT = selectWithinFam(HDRW, nInd=fam_max, simParam=SP)
    PYT = selectInd(PYT, nInd=n_PYT, simParam=SP)
    PYT = setPheno(PYT, varE=var_e, reps=rep_PYT, simParam=SP)
    
    # Stage 3: HDRW
    HDRW = setPheno(DH, varE=var_e, reps=rep_HDRW, simParam=SP)
    
    # Stage 2: DH lines from F1 (using makeDH - key difference from SSD)
    DH = makeDH(F1, nDH=n_dh, simParam=SP)
    
    # Stage 1: F1 crosses
    F1 = randCross(Parents, n_crosses, simParam=SP)
    
    return F1, DH, HDRW, PYT, AYT, EYT

print("Advance year function defined.")


## Update Parents Function

This function mirrors `UpdateParents.R`. It replaces the 10 oldest parents (indices 0-9) with the 10 new parents from the EYT stage, maintaining a constant population size of `n_parents`.


In [ ]:
def update_parents(Parents, EYT):
    """
    Update parents by replacing oldest 10 with new EYT entries.
    
    Mirrors UpdateParents.R: Parents = c(Parents[11:nParents], EYT)
    In Python, we select indices 10 onwards (0-indexed) and merge with EYT.
    """
    # Select parents from index 10 onwards (keeping n_parents - n_EYT individuals)
    keep_indices = list(range(n_EYT, n_parents))
    if len(keep_indices) > 0:
        Parents_keep = selectInd(Parents, nInd=len(keep_indices), candidates=keep_indices, simParam=SP)
        # Merge kept parents with new EYT entries
        Parents_new = mergePops([Parents_keep, EYT])
    else:
        Parents_new = EYT
    
    return Parents_new

print("Update parents function defined.")


## Burn-in Phase

Run the burn-in phase to establish the breeding program. This mirrors the burn-in loop in `00RUNME.R`, tracking mean genetic value, genetic variance, and selection accuracy each year.


In [ ]:
print("Starting burn-in phase...")

# Initialize output tracking (mirrors output data.frame in 00RUNME.R)
output = {
    'year': list(range(1, n_cycles + 1)),
    'rep': [1] * n_cycles,
    'scenario': ['LinePheno_DH'] * n_cycles,
    'meanG': [0.0] * n_cycles,
    'varG': [0.0] * n_cycles,
    'accSel': [0.0] * n_cycles,
}

# Burn-in phase
for year in range(1, n_burnin + 1):
    print(f"  Working on burn-in year: {year}")
    
    # Update parents (pick new parents from EYT)
    Parents = update_parents(Parents, EYT)
    
    # Advance breeding program by 1 year
    F1, DH, HDRW, PYT, AYT, EYT = advance_year(
        Parents, F1, DH, HDRW, PYT, AYT, EYT, year, output
    )
    
    # Report results (using DH as in R)
    output['meanG'][year-1] = meanG(DH)[0]
    output['varG'][year-1] = varG(DH)[0]
    
    if year % 5 == 0:
        print(f"    Year {year}: Mean G = {output['meanG'][year-1]:.3f}, "
              f"Var G = {output['varG'][year-1]:.3f}, "
              f"Acc = {output['accSel'][year-1]:.3f}")

print(f"\nBurn-in phase completed!")
print(f"  Final mean G: {output['meanG'][n_burnin-1]:.3f}")
print(f"  Final var G: {output['varG'][n_burnin-1]:.3f}")
print(f"  Final accuracy: {output['accSel'][n_burnin-1]:.3f}")


## Future Phase

Continue the breeding program in the future phase, tracking the same metrics.


In [ ]:
print("Starting future phase...")

# Future phase
for year in range(n_burnin + 1, n_burnin + n_future + 1):
    print(f"  Working on future year: {year}")
    
    # Update parents (pick new parents from EYT)
    Parents = update_parents(Parents, EYT)
    
    # Advance breeding program by 1 year
    F1, DH, HDRW, PYT, AYT, EYT = advance_year(
        Parents, F1, DH, HDRW, PYT, AYT, EYT, year, output
    )
    
    # Report results (using DH as in R)
    output['meanG'][year-1] = meanG(DH)[0]
    output['varG'][year-1] = varG(DH)[0]
    
    if (year - n_burnin) % 5 == 0:
        print(f"    Year {year}: Mean G = {output['meanG'][year-1]:.3f}, "
              f"Var G = {output['varG'][year-1]:.3f}, "
              f"Acc = {output['accSel'][year-1]:.3f}")

print(f"\nFuture phase completed!")
print(f"  Final mean G: {output['meanG'][n_cycles-1]:.3f}")
print(f"  Final var G: {output['varG'][n_cycles-1]:.3f}")
print(f"  Final accuracy: {output['accSel'][n_cycles-1]:.3f}")


## Analyze and Plot Results

This section mirrors `ANALYZERESULTS.R`, creating plots of genetic gain, genetic variance, and selection accuracy over time.


In [ ]:
# Convert output to numpy arrays for easier plotting
years = np.array(output['year'])
mean_g = np.array(output['meanG'])
var_g = np.array(output['varG'])
acc_sel = np.array(output['accSel'])

# Create figure with 3 subplots (mirrors ANALYZERESULTS.R)
fig, axes = plt.subplots(3, 1, figsize=(6, 10))
fig.suptitle('Doubled Haploid Breeding Program Results', fontsize=14, fontweight='bold')

# Plot 1: Genetic Gain
axes[0].plot(years, mean_g, 'b-', linewidth=2, label='Mean Genetic Value')
axes[0].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Yield')
axes[0].set_title('Genetic Gain')
axes[0].grid(True, linestyle='--', alpha=0.3)
axes[0].legend()

# Plot 2: Genetic Variance
axes[1].plot(years, var_g, 'b-', linewidth=2, label='Genetic Variance')
axes[1].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Variance')
axes[1].set_title('Genetic Variance')
axes[1].grid(True, linestyle='--', alpha=0.3)
axes[1].legend()

# Plot 3: Selection Accuracy
axes[2].plot(years, acc_sel, 'b-', linewidth=2, label='Selection Accuracy')
axes[2].axvline(x=n_burnin, color='r', linestyle='--', alpha=0.5, label='Burn-in/Future')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Correlation')
axes[2].set_title('Selection Accuracy')
axes[2].grid(True, linestyle='--', alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig('DoubledHaploid_Results.png', dpi=150, bbox_inches='tight')
print("Results plotted and saved to DoubledHaploid_Results.png")
plt.show()

# Print summary statistics
print("\nSummary Statistics:")
print(f"  Initial mean G: {mean_g[0]:.3f}")
print(f"  Final mean G: {mean_g[-1]:.3f}")
print(f"  Total genetic gain: {mean_g[-1] - mean_g[0]:.3f}")
print(f"  Initial var G: {var_g[0]:.3f}")
print(f"  Final var G: {var_g[-1]:.3f}")
print(f"  Mean selection accuracy: {np.nanmean(acc_sel):.3f}")


## Summary

This tutorial demonstrated a **doubled haploid** breeding program using AlphaSimPy, which:

1. **Uses DH technology** (`makeDH()`) to create homozygous lines instantly from F1 crosses, eliminating the need for multiple generations of selfing
2. **Progresses through yield trial stages** (HDRW → PYT → AYT → EYT) with increasing replication and heritability
3. **Selects within families** before individual selection, maintaining family structure
4. **Updates parents** each year by replacing the oldest parents with new elite entries from EYT

Key differences from SSD (Single Seed Descent):
- **Instant homozygosity**: DH lines are homozygous immediately, whereas SSD requires multiple generations of selfing
- **Faster breeding cycle**: No need to wait for multiple selfing generations
- **Different technology**: Uses `makeDH()` instead of `self()` to create homozygous lines

The results show genetic gain over time, changes in genetic variance, and selection accuracy throughout the breeding program.
